In [1]:
# Ambiente e percorsi
from pathlib import Path
import sys
import pandas as pd
import yaml

cwd = Path.cwd()
if (cwd / "generate_dataset.py").exists():
    DATA_AGENT_DIR = cwd
elif (cwd.parent / "generate_dataset.py").exists():
    DATA_AGENT_DIR = cwd.parent
else:
    DATA_AGENT_DIR = cwd / "data_agent"

CONFIG_DIR = DATA_AGENT_DIR / "config"
if str(DATA_AGENT_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_AGENT_DIR))

print("Data Agent:", DATA_AGENT_DIR)


Data Agent: C:\Users\user\Desktop\Master AI e Agenti AI per il Business\30 Sviluppa un agente AI avanzato\TrialPulse\data_agent


In [2]:
# Controllo configurazioni
with open(CONFIG_DIR / "study_design.yaml", encoding="utf-8") as f:
    study_design = yaml.safe_load(f)
with open(CONFIG_DIR / "study_profiles.yaml", encoding="utf-8") as f:
    study_profiles = yaml.safe_load(f)
with open(CONFIG_DIR / "synthetic_generation.yaml", encoding="utf-8") as f:
    generation_config = yaml.safe_load(f)
with open(CONFIG_DIR / "plausibility_ranges.yaml", encoding="utf-8") as f:
    plausibility = yaml.safe_load(f)

study_ids = set(study_design["studies"])
assert study_ids == set(study_profiles["studies"])
assert study_ids == set(plausibility["efficacy"])
assert study_ids == set(plausibility["safety"])
for study_id, study in study_design["studies"].items():
    allocation = study_design["site_allocation"][study_id]
    assert sum(x["screened"] for x in allocation.values()) == study["screened"]
    assert sum(x["enrolled"] for x in allocation.values()) == study["enrolled"]
    assert len(study_design["schedule"][study_id]) == study["visits"]

print("Configurazioni validate correttamente.")


Configurazioni validate correttamente.


In [3]:
# Generazione dataset
from generate_dataset import generate_dataset, MASTER_SEED

clean, dirty = generate_dataset()
print("Seed:", MASTER_SEED)
print("Clean:", clean.shape, "| Dirty:", dirty.shape)
print("Duplicati dirty:", dirty.duplicated().sum())


Seed: 20261003
Clean: (1500, 43) | Dirty: (1508, 43)
Duplicati dirty: 8


In [4]:
# QA operativo
eligible = clean.query(
    "screening_failure == False and visit_name != 'Screening' "
    "and visit_status != 'not_expected_after_dropout'"
)
missed = eligible.assign(
    missed=eligible["visit_status"].eq("missed")
).groupby("site_id")["missed"].mean().mul(100).round(1)

queries = clean[clean["data_queries_open"] > 0].groupby(
    "site_id"
)["oldest_open_query_days"].mean().round(1)

assert missed.idxmax() == "SITE_07"
assert queries.idxmax() == "SITE_05"
print("Missed visit per sito (%)")
display(missed.sort_values(ascending=False))
print("Query aging medio per sito")
display(queries.sort_values(ascending=False))


Missed visit per sito (%)


site_id
SITE_07    13.4
SITE_03     5.5
SITE_04     4.0
SITE_05     3.2
SITE_06     2.7
SITE_01     1.9
SITE_08     1.8
SITE_02     1.7
Name: missed, dtype: float64

Query aging medio per sito


site_id
SITE_05    24.5
SITE_06    13.1
SITE_07     9.1
SITE_03     8.2
SITE_01     7.9
SITE_04     7.7
SITE_02     5.8
SITE_08     4.7
Name: oldest_open_query_days, dtype: float64

In [5]:
# QA efficacia e safety
rows = []
for study_id, cfg in plausibility["efficacy"].items():
    endpoint = clean[
        (clean["study_id"] == study_id)
        & clean["visit_name"].eq(cfg["endpoint_visit"])
        & clean["visit_status"].eq("completed")
    ]
    rows.append({
        "study_id": study_id,
        "endpoint": cfg["endpoint_visit"],
        "score_mean": round(endpoint["depression_score"].mean(), 2),
        "change_mean": round(endpoint["depression_change_from_baseline"].mean(), 2),
        "response_%": round(endpoint["clinical_response"].astype(bool).mean() * 100, 1),
        "remission_%": round(endpoint["clinical_remission"].astype(bool).mean() * 100, 1),
    })

efficacy_summary = pd.DataFrame(rows)
display(efficacy_summary)

safety_summary = clean[~clean["screening_failure"]].groupby(
    ["study_id", "participant_id"]
)[["adverse_event", "serious_adverse_event", "ae_leading_to_discontinuation"]].max().groupby("study_id").mean().mul(100).round(1)
display(safety_summary)


,study_id,endpoint,score_mean,change_mean,response_%,remission_%
0,NNR-PSI-01,Week 3,19.82,-13.21,36.4,30.3
1,NNR-ESK-02,Week 8,16.50,-14.80,50.0,26.7
2,NNR-DXB-03,Week 6,16.73,-16.83,53.7,39.0
3,NNR-SER-04,Week 6,7.56,-5.69,52.8,63.9


,adverse_event,serious_adverse_event,ae_leading_to_discontinuation
study_id,,,
NNR-DXB-03,61.4,2.3,4.5
NNR-ESK-02,90.6,6.2,3.1
NNR-PSI-01,83.3,5.6,2.8
NNR-SER-04,42.9,2.4,4.8


In [6]:
# QA dirty data
assert len(clean) == 1500
assert len(dirty) == 1508
assert clean.shape[1] == 43
assert dirty.duplicated().sum() == 8
assert clean["record_id"].is_unique
assert dirty[["record_id", "study_id", "site_id", "participant_id"]].notna().all().all()

print("QA dataset superato.")


QA dataset superato.
